In [1]:
# Task 1.1 — Read the file the foundations way (5 marks)

# Using only the csv module (not pandas): open appointments.csv with with open(..., newline=""), read it
# with csv.DictReader, and print the header and row count. Write a function parse_row(row) that converts age, fee_inr and prior_no_shows to int and distance_km to float, returning the converted dictionary. Loop over every row; use try/except ValueError to put rows that fail into a rejected list and the rest into an accepted list. Save rejected with json.dump(..., indent=2) as rejected_rows.json.

# Solution
# ==========================================
# CARELOOP CLINICS
# PART 1 - TASK 1.1
# Read the file the foundations way
# ==========================================
# Step 1: Import the required modules
import csv
import json

# Explanation:
# csv reads CSV files.
# json saves rejected records in JSON format

# Step 2: Open and read the CSV file
# Understand this structure:

with open("appointments.csv", "r", newline="",
          encoding="utf-8") as file:

    reader = csv.DictReader(file)

    rows = list(reader)

    print("Headers:", reader.fieldnames)
    print("Total rows:", len(rows))

# Explanation:
#        Code                                              Meaning
#       open()                                       Opens the CSV file
#      newline=""                              Handles CSV newlines correctly
#     DictReader()                             Reads each row as a dictionary
#      list(reader)                                Stores all rows in a list
#      fieldnames                                     Displays column names
#       len(rows)                                         Counts records

# Step 3: Understand the parse_row(row) function
def parse_row(row):
    row["age"] =int(row["age"])
    return row

# Step 4: Separate accepted and rejected rows

accepted = []
rejected = []

for row in rows:

    try:
        converted_row = parse_row(row)
        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)

# Step 5: Save rejected rows as JSON
with open("rejected_rows.json", "w",
          encoding="utf-8") as file:

    json.dump(rejected, file, indent=2)

# Step 6: Check  results

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))
print("Total rows:", len(accepted) + len(rejected))

print("\nTask 1.1 completed successfully!")

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020
Accepted rows: 1008
Rejected rows: 12
Total rows: 1020

Task 1.1 completed successfully!


In [2]:
# Task 1.2 — Inspect and clean, pass 1 (5 marks)

# Load the file with pd.read_csv. Print shape, info(), isnull().sum() and duplicated().sum(). Then, in this order: (a) drop exact duplicate rows; (b) clean booking_channel with .str.strip().str.title(); (c) convert appt_date with pd.to_datetime(..., errors="coerce") and booked_on with pd.to_datetime; (d) drop rows where appt_date is missing after conversion or status is missing. Print the row count after each step. This task starts again from the original appointments.csv: Task 1.1 is a separate exercise in reading with the  csv  module, so do not use its accepted/rejected lists here — the pandas cleaning in 1.2–1.3 is the one that  feeds the rest of the project.

# Solution
# ==========================================
# CARELOOP CLINICS
# PART 1 - TASK 1.2
# Inspect and clean, pass 1
# ==========================================
from pathlib import Path
import pandas as pd
# ==========================================
# STEP 1: LOAD ORIGINAL CSV
# ==========================================
file_path = Path("appointments.csv")

df = pd.read_csv(file_path)

print("========== STEP 1: ORIGINAL DATA ==========")

print("Original shape:", df.shape)
print("Original row count:", len(df))

# Step 2: Inspect the dataset
print("\n========== STEP 2: DATA INFORMATION ==========")

print("\nDataset information:")
df.info()

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nNumber of exact duplicate rows:")
print(df.duplicated().sum())

# Step 3: Drop exact duplicate rows
print("\n========== STEP 3: DROP DUPLICATES ==========")

df = df.drop_duplicates().copy()

print("Rows after dropping duplicates:", len(df))

# Step 4: Clean booking_channel
print("\n========== STEP 4: CLEAN BOOKING CHANNEL ==========")

df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

print("Rows after cleaning booking_channel:", len(df))

print("\nBooking channel values:")
print(df["booking_channel"].value_counts(dropna=False))

# Step 5: Convert date columns into datetime format
print("\n========== STEP 5: CONVERT DATE COLUMNS ==========")

df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"], errors="coerce"
)

print("Rows after date conversion:", len(df))

print("\nDate column data types:")
print(df[["appt_date", "booked_on"]].dtypes)

    # Step 6: Drop rows with missing appointment date or status
print("\n========== STEP 6: DROP MISSING VALUES ==========")

before_drop = len(df)

df = df.dropna(subset=["appt_date", "status"]).copy()

after_drop = len(df)

print("Rows before dropping missing values:", before_drop)
print("Rows removed:", before_drop - after_drop)
print("Final rows after cleaning:", after_drop)

# Step 7: Final dataset information
print("\n========== FINAL CLEANED DATA ==========")

print("Final shape:", df.shape)

print("\nRemaining missing values:")
print(df.isnull().sum())

print("\nTask 1.2 completed successfully!")

========== STEP 1: ORIGINAL DATA ==========
Original shape: (1020, 14)
Original row count: 1020

========== STEP 2: DATA INFORMATION ==========

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dt

In [4]:
# Task 1.3 — Clean, pass 2: validity, gaps and outliers (5 marks)

# Work in this order, so that the quartiles and fences in (c) are computed after the gaps are filled. (a)
# Treat any age below 1 or above 100 as invalid and set it to missing. (b) Fill missing age with the median (then .round().astype(int)) and missing distance_km with its median. (c) For distance_km and fee_inr, compute Q1, Q3 and IQR with .quantile, and cap values at Q3 + 1.5×IQR and Q1 − 1.5×IQR with np.clip. (d) Round distance_km to 1 decimal and fee_inr to whole rupees (.round().astype(int)). Print how many values  you capped in each column.

# Soluiton
# ==========================================
# CARELOOP CLINICS
# PART 1 - TASK 1.3
# Clean, pass 2: validity, gaps and outliers
# ==========================================
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# STEP 1: LOAD ORIGINAL CSV
# ==========================================

file_path = Path("appointments.csv")

df = pd.read_csv(file_path)

print("========== TASK 1.3 ==========")
print("Original rows:", len(df))

# ==========================================
# REPEAT TASK 1.2 CLEANING
# ==========================================

# Remove exact duplicates
df = df.drop_duplicates().copy()

print("Rows after removing duplicates:", len(df))

# Clean booking_channel
df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

# Convert date columns
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"], errors="coerce"
)

# Remove missing appointment date and status
df = df.dropna(subset=["appt_date", "status"]).copy()

print("Rows after Task 1.2 cleaning:", len(df))


# ==========================================
# STEP 2: VALIDATE AGE
# ==========================================

print("\n========== STEP 2: AGE VALIDATION ==========")

df["age"] = pd.to_numeric(
    df["age"], errors="coerce"
)

# Find invalid ages
invalid_age = (
    (df["age"] < 1) | (df["age"] > 100)
)

print("Invalid age values:", invalid_age.sum())

# Replace invalid ages with missing values
df.loc[invalid_age, "age"] = np.nan


# ==========================================
# STEP 3: FILL MISSING VALUES
# ==========================================

print("\n========== STEP 3: FILL MISSING VALUES ==========")

# Fill missing age with median
age_median = df["age"].median()

df["age"] = (
    df["age"]
    .fillna(age_median)
    .round()
    .astype(int)
)

print("Age median:", age_median)
print("Missing age after filling:", df["age"].isna().sum())

# Fill missing distance with median
df["distance_km"] = pd.to_numeric(
    df["distance_km"], errors="coerce"
)

distance_median = df["distance_km"].median()

df["distance_km"] = df["distance_km"].fillna(
    distance_median
)

print("Distance median:", distance_median)
print(
    "Missing distance after filling:",
    df["distance_km"].isna().sum()
)


# ==========================================
# STEP 4: DISTANCE OUTLIERS USING IQR
# ==========================================

print("\n========== STEP 4: DISTANCE OUTLIERS ==========")

Q1_distance = df["distance_km"].quantile(0.25)
Q3_distance = df["distance_km"].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = Q1_distance - 1.5 * IQR_distance
upper_distance = Q3_distance + 1.5 * IQR_distance

print("Distance Q1:", Q1_distance)
print("Distance Q3:", Q3_distance)
print("Distance IQR:", IQR_distance)

print("Distance lower limit:", lower_distance)
print("Distance upper limit:", upper_distance)

# Count values outside the limits BEFORE capping
distance_capped = (
    (df["distance_km"] < lower_distance)
    | (df["distance_km"] > upper_distance)
)

print("Distance values capped:", distance_capped.sum())

# Cap distance values
df["distance_km"] = np.clip(
    df["distance_km"],
    lower_distance,
    upper_distance
)


# ==========================================
# STEP 5: FEE OUTLIERS USING IQR
# ==========================================

print("\n========== STEP 5: FEE OUTLIERS ==========")

df["fee_inr"] = pd.to_numeric(
    df["fee_inr"], errors="coerce"
)

Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

print("Fee Q1:", Q1_fee)
print("Fee Q3:", Q3_fee)
print("Fee IQR:", IQR_fee)

print("Fee lower limit:", lower_fee)
print("Fee upper limit:", upper_fee)

# Count values outside the limits BEFORE capping
fee_capped = (
    (df["fee_inr"] < lower_fee)
    | (df["fee_inr"] > upper_fee)
)

print("Fee values capped:", fee_capped.sum())

# Cap fee values
df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fee,
    upper_fee
)


# ==========================================
# STEP 6: ROUND VALUES
# ==========================================

print("\n========== STEP 6: ROUND VALUES ==========")

# Distance to 1 decimal place
df["distance_km"] = df["distance_km"].round(1)

# Fee to whole rupees
df["fee_inr"] = (
    df["fee_inr"].round().astype(int)
)

print("Distance rounded to 1 decimal place.")
print("Fee rounded to whole rupees.")


# ==========================================
# STEP 7: FINAL RESULTS
# ==========================================

print("\n========== FINAL RESULTS ==========")

print("Final dataset shape:", df.shape)

print("Final age data type:", df["age"].dtype)
print("Final distance data type:", df["distance_km"].dtype)
print("Final fee data type:", df["fee_inr"].dtype)

print("\nTotal distance values capped:", distance_capped.sum())
print("Total fee values capped:", fee_capped.sum())

print("\nTask 1.3 completed successfully!")

========== TASK 1.3 ==========
Original rows: 1020
Rows after removing duplicates: 1000
Rows after Task 1.2 cleaning: 990

========== STEP 2: AGE VALIDATION ==========
Invalid age values: 5

========== STEP 3: FILL MISSING VALUES ==========
Age median: 48.0
Missing age after filling: 0
Distance median: 8.25
Missing distance after filling: 0

========== STEP 4: DISTANCE OUTLIERS ==========
Distance Q1: 4.625
Distance Q3: 13.0
Distance IQR: 8.375
Distance lower limit: -7.9375
Distance upper limit: 25.5625
Distance values capped: 26

========== STEP 5: FEE OUTLIERS ==========
Fee Q1: 500.0
Fee Q3: 787.5
Fee IQR: 287.5
Fee lower limit: 68.75
Fee upper limit: 1218.75
Fee values capped: 3

========== STEP 6: ROUND VALUES ==========
Distance rounded to 1 decimal place.
Fee rounded to whole rupees.

========== FINAL RESULTS ==========
Final dataset shape: (990, 14)
Final age data type: int64
Final distance data type: float64
Final fee data type: int64

Total distance values capped: 26
Total fe

In [5]:
# Task 1.4 — Features and a NumPy summary (4 marks)

# Create lead_days = days between booked_on and appt_date (use .dt.days) and no_show = 1 if status is
# "No-Show" else 0. Then take X = df[["age","distance_km","fee_inr","lead_days"]].to_numpy() and, without
# loops, compute: (a) the mean of each column (axis=0); (b) the number and share of rows with lead_days > 14 using a boolean mask (mask.sum() counts the True values and mask.mean() gives the share); (c) z-scores of the distance_km column using (x - np.mean(x)) / np.std(x) and the count of rows with |z| > 2.

# Solution
# ==========================================
# CARELOOP CLINICS
# PART 1 - TASK 1.4
# Features and a NumPy summary
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# STEP 1: LOAD ORIGINAL CSV
# ==========================================

file_path = Path("appointments.csv")

df = pd.read_csv(file_path)

print("========== TASK 1.4 ==========")
print("Original rows:", len(df))

# ==========================================
# STEP 2: REPEAT TASK 1.2 CLEANING
# ==========================================

# Remove duplicate rows
df = df.drop_duplicates().copy()

# Clean booking channel
df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

# Convert dates
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"], errors="coerce"
)

# Remove missing appointment date and status
df = df.dropna(subset=["appt_date", "status"]).copy()

print("\nRows after Task 1.2 cleaning:", len(df))


# ==========================================
# STEP 3: REPEAT TASK 1.3 CLEANING
# ==========================================

# Validate age
df["age"] = pd.to_numeric(
    df["age"], errors="coerce"
)

invalid_age = (
    (df["age"] < 1) | (df["age"] > 100)
)

df.loc[invalid_age, "age"] = np.nan

# Fill missing age with median
age_median = df["age"].median()

df["age"] = (
    df["age"]
    .fillna(age_median)
    .round()
    .astype(int)
)

# Convert distance to numeric
df["distance_km"] = pd.to_numeric(
    df["distance_km"], errors="coerce"
)

# Fill missing distance with median
distance_median = df["distance_km"].median()

df["distance_km"] = df["distance_km"].fillna(
    distance_median
)

# Convert fee to numeric
df["fee_inr"] = pd.to_numeric(
    df["fee_inr"], errors="coerce"
)

# Calculate distance IQR
Q1_distance = df["distance_km"].quantile(0.25)
Q3_distance = df["distance_km"].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = Q1_distance - 1.5 * IQR_distance
upper_distance = Q3_distance + 1.5 * IQR_distance

# Cap distance outliers
df["distance_km"] = np.clip(
    df["distance_km"],
    lower_distance,
    upper_distance
)

# Calculate fee IQR
Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

# Cap fee outliers
df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fee,
    upper_fee
)

# Round values
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = (
    df["fee_inr"].round().astype(int)
)

print("Rows after Task 1.3 cleaning:", len(df))


# ==========================================
# STEP 4: CREATE LEAD_DAYS FEATURE
# ==========================================

print("\n========== STEP 4: CREATE FEATURES ==========")

df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

print("Lead days feature created successfully.")

# ==========================================
# STEP 5: CREATE NO_SHOW FEATURE
# ==========================================

df["no_show"] = (
    df["status"] == "No-Show"
).astype(int)

print("No-show feature created successfully.")

print("\nNo-show counts:")
print(df["no_show"].value_counts())


# ==========================================
# STEP 6: CREATE NUMPY ARRAY
# ==========================================

print("\n========== STEP 6: NUMPY ARRAY ==========")

X = df[
    ["age", "distance_km", "fee_inr", "lead_days"]
].to_numpy()

print("NumPy array:")
print(X)

print("\nArray shape:", X.shape)


# ==========================================
# STEP 7: CALCULATE COLUMN MEANS
# ==========================================

print("\n========== STEP 7: COLUMN MEANS ==========")

column_means = np.mean(X, axis=0)

print("Mean of each column:")
print(column_means)

print("Column order: age, distance_km, fee_inr, lead_days")


# ==========================================
# STEP 8: LEAD DAYS GREATER THAN 14
# ==========================================

print("\n========== STEP 8: LEAD DAYS ANALYSIS ==========")

mask = df["lead_days"] > 14

# Count rows where lead_days > 14
lead_count = mask.sum()

# Calculate share of rows
lead_share = mask.mean()

print("Rows with lead_days > 14:", lead_count)

print("Share of rows:", lead_share)

print("Percentage of rows:", lead_share * 100)


# ==========================================
# STEP 9: DISTANCE Z-SCORES
# ==========================================

print("\n========== STEP 9: DISTANCE Z-SCORES ==========")

x = X[:, 1]

z = (x - np.mean(x)) / np.std(x)

print("Distance Z-scores:")
print(z)

# Count rows where absolute Z-score > 2
z_count = np.sum(np.abs(z) > 2)

print("Rows with absolute Z-score > 2:", z_count)


# ==========================================
# STEP 10: FINAL RESULTS
# ==========================================

print("\n========== FINAL RESULTS ==========")

print("Final dataset shape:", df.shape)

print("Total no-show records:", df["no_show"].sum())

print("Total rows with lead_days > 14:", lead_count)

print("Total distance Z-score values above 2:", z_count)

print("\nTask 1.4 completed successfully!")

========== TASK 1.4 ==========
Original rows: 1020

Rows after Task 1.2 cleaning: 990
Rows after Task 1.3 cleaning: 990

========== STEP 4: CREATE FEATURES ==========
Lead days feature created successfully.
No-show feature created successfully.

No-show counts:
no_show
0    772
1    218
Name: count, dtype: int64

========== STEP 6: NUMPY ARRAY ==========
NumPy array:
[[ 60.   18.  850.    1. ]
 [ 39.   12.5 350.   10. ]
 [ 63.    4.1 850.    6. ]
 ...
 [ 58.    3.2 850.   11. ]
 [ 59.   25.6 300.   13. ]
 [ 38.   10.8 550.   45. ]]

Array shape: (990, 4)

========== STEP 7: COLUMN MEANS ==========
Mean of each column:
[ 48.17373737   9.50090909 635.71414141  11.81919192]
Column order: age, distance_km, fee_inr, lead_days

========== STEP 8: LEAD DAYS ANALYSIS ==========
Rows with lead_days > 14: 296
Share of rows: 0.298989898989899
Percentage of rows: 29.898989898989896

========== STEP 9: DISTANCE Z-SCORES ==========
Distance Z-scores:
[ 1.38034946  0.48708663 -0.87716934 -0.3249705  

In [6]:
# Task 1.5 — Enrich the clinics from a live API (3 marks)

# For each of the 8 clinics call the free Open-Meteo API with requests.get
#("https://api.open-meteo.com/v1/forecast", params={"latitude": ..., "longitude": ..., "hourly":
# "temperature_2m",  "forecast_days": 1}). Check status_code == 200 before using the reply, read the JSON,take the first value of ["hourly"]["temperature_2m"] (the midnight-UTC reading, a stand-in rather than the current  weather) as temp_c, and save clinics_enriched.csv (the original columns plus temp_c;
# df.to_csv("file.csv", index=False) writes a DataFrame to CSV). No API key is needed.
# If the API fails: pass timeout=10 to requests.get, wrap the call in try/except requests.exceptions.
# RequestException, retry that clinic once, and if it still fails (or the status is not 200, or hourly
# is missing) store None as temp_c, print the clinic('s name and the reason, and carry on. Never type in a ' ,made-up temperature; in load_data.sql a missing value is written as NULL (unquoted).)

# Solution
# ==========================================
# CARELOOP CLINICS
# PART 1 - TASK 1.5
# Enrich the clinics from a live API
# ==========================================

import pandas as pd
import requests

# Read the original clinics CSV
clinics_df = pd.read_csv("clinics.csv")

# Check required columns
print("Available columns:", clinics_df.columns.tolist())

# Prepare a copy of clinics data
clinics_enriched = clinics_df.copy()

# Store temperature values
temperatures = []

# Loop through all clinics
for index, clinic in clinics_enriched.iterrows():

    clinic_name = clinic.get("clinic_name", f"Clinic {index + 1}")
    latitude = clinic["latitude"]
    longitude = clinic["longitude"]

    temp_c = None
    reason = None

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    # Retry once only for request exceptions
    for attempt in range(2):

        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                reason = f"HTTP status code: {response.status_code}"
                break

            weather_data = response.json()

            hourly = weather_data.get("hourly", {})
            temp_values = hourly.get("temperature_2m")

            if not temp_values:
                reason = "Hourly temperature data is missing"
                break

            temp_c = temp_values[0]
            reason = None
            break

        except requests.exceptions.RequestException as error:

            reason = str(error)

            if attempt == 0:
                print(f"Retrying {clinic_name}...")

    # Append result
    temperatures.append(temp_c)

    # Print failure reason
    if temp_c is None:
        print(f"Failed for {clinic_name}: {reason}")

# Add temperature column
clinics_enriched["temp_c"] = temperatures

# Save output CSV
clinics_enriched.to_csv("clinics_enriched.csv", index=False)

# Display results
print("\nEnriched Clinics Data:")
print(clinics_enriched)

print("\nFile saved: clinics_enriched.csv")


print("========== TASK 1.4 ==========")
print("Original rows:", len(df))

# ==========================================
# STEP 2: REPEAT TASK 1.2 CLEANING
# ==========================================

# Remove duplicate rows
df = df.drop_duplicates().copy()

# Clean booking channel
df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

# Convert dates
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"], errors="coerce"
)

# Remove missing appointment date and status
df = df.dropna(subset=["appt_date", "status"]).copy()

print("\nRows after Task 1.2 cleaning:", len(df))


# ==========================================
# STEP 3: REPEAT TASK 1.3 CLEANING
# ==========================================

# Validate age
df["age"] = pd.to_numeric(
    df["age"], errors="coerce"
)

invalid_age = (
    (df["age"] < 1) | (df["age"] > 100)
)

df.loc[invalid_age, "age"] = np.nan

# Fill missing age with median
age_median = df["age"].median()

df["age"] = (
    df["age"]
    .fillna(age_median)
    .round()
    .astype(int)
)

# Convert distance to numeric
df["distance_km"] = pd.to_numeric(
    df["distance_km"], errors="coerce"
)

# Fill missing distance with median
distance_median = df["distance_km"].median()

df["distance_km"] = df["distance_km"].fillna(
    distance_median
)

# Convert fee to numeric
df["fee_inr"] = pd.to_numeric(
    df["fee_inr"], errors="coerce"
)

# Calculate distance IQR
Q1_distance = df["distance_km"].quantile(0.25)
Q3_distance = df["distance_km"].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = Q1_distance - 1.5 * IQR_distance
upper_distance = Q3_distance + 1.5 * IQR_distance

# Cap distance outliers
df["distance_km"] = np.clip(
    df["distance_km"],
    lower_distance,
    upper_distance
)

# Calculate fee IQR
Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

# Cap fee outliers
df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fee,
    upper_fee
)

# Round values
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = (
    df["fee_inr"].round().astype(int)
)

print("Rows after Task 1.3 cleaning:", len(df))


# ==========================================
# STEP 4: CREATE LEAD_DAYS FEATURE
# ==========================================

print("\n========== STEP 4: CREATE FEATURES ==========")

df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

print("Lead days feature created successfully.")

# ==========================================
# STEP 5: CREATE NO_SHOW FEATURE
# ==========================================

df["no_show"] = (
    df["status"] == "No-Show"
).astype(int)

print("No-show feature created successfully.")

print("\nNo-show counts:")
print(df["no_show"].value_counts())


# ==========================================
# STEP 6: CREATE NUMPY ARRAY
# ==========================================

print("\n========== STEP 6: NUMPY ARRAY ==========")

X = df[
    ["age", "distance_km", "fee_inr", "lead_days"]
].to_numpy()

print("NumPy array:")
print(X)

print("\nArray shape:", X.shape)


# ==========================================
# STEP 7: CALCULATE COLUMN MEANS
# ==========================================

print("\n========== STEP 7: COLUMN MEANS ==========")

column_means = np.mean(X, axis=0)

print("Mean of each column:")
print(column_means)

print("Column order: age, distance_km, fee_inr, lead_days")


# ==========================================
# STEP 8: LEAD DAYS GREATER THAN 14
# ==========================================

print("\n========== STEP 8: LEAD DAYS ANALYSIS ==========")

mask = df["lead_days"] > 14

# Count rows where lead_days > 14
lead_count = mask.sum()

# Calculate share of rows
lead_share = mask.mean()

print("Rows with lead_days > 14:", lead_count)

print("Share of rows:", lead_share)

print("Percentage of rows:", lead_share * 100)


# ==========================================
# STEP 9: DISTANCE Z-SCORES
# ==========================================

print("\n========== STEP 9: DISTANCE Z-SCORES ==========")

x = X[:, 1]

z = (x - np.mean(x)) / np.std(x)

print("Distance Z-scores:")
print(z)

# Count rows where absolute Z-score > 2
z_count = np.sum(np.abs(z) > 2)

print("Rows with absolute Z-score > 2:", z_count)


# ==========================================
# STEP 10: FINAL RESULTS
# ==========================================

print("\n========== FINAL RESULTS ==========")

print("Final dataset shape:", df.shape)

print("Total no-show records:", df["no_show"].sum())

print("Total rows with lead_days > 14:", lead_count)

print("Total distance Z-score values above 2:", z_count)

print("\nTask 1.5 completed successfully!")

Available columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']

Enriched Clinics Data:
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    18.3  
1    20.0  
2    20.4  
3    16.2  
4    16.2  
5    26.2  
6    16.6  
7    16.9  

File saved: clinics_enriched.csv
========== TASK 1.4 ==========
Or

In [7]:
# Task 1.6 — Export the clean table and write the SQL load script (4 marks)

# Sort the cleaned data by appointment_id, format both date columns as YYYY-MM-DD text (.dt.strftime
# ("%Y-%m-%d")), and save appointments_clean.csv with exactly these 16 columns, in this order: appointment_id, patient_id, clinic_id, booked_on, appt_date, age, gender, distance_km, fee_inr, booking_channel, visit_type,reminder_sent, prior_no_shows, status, lead_days, no_show. Then, with a for loop and f-strings, write load_data.sql containing one statement per row, in this pattern (text in single quotes, numbers unquoted): INSERT INTO appointments VALUES ('A0001','P196','C05','2025-01-22',...,0); — and likewise one INSERT INTO clinics VALUES (...) per clinic from clinics_enriched.csv. Write all the clinics statements first, because the appointments table points to them with a foreign key. Before exporting, check and print that appointment_id is unique and that none of the 16 columns has a missing value. (Building SQL text with f-strings is fine for this fixed, controlled dataset — none of its values contains an apostrophe. It is not a safe method for real applications, where parameterised queries are used instead.)

# Solution
# ==========================================
# CARELOOP CLINICS
# PART 1 - TASK 1.6
# Export the clean table and write the SQL load script
# ==========================================
import pandas as pd
import numpy as np
from pathlib import Path
import numbers

# ==========================================
# STEP 1: LOAD ORIGINAL APPOINTMENT DATA
# ==========================================

from pathlib import Path
folder = Path.cwd()

input_file = folder / "appointments.csv"
clinic_file = folder / "clinics_enriched.csv"

df = pd.read_csv(input_file)

print("========== TASK 1.6 ==========")
print("Original rows:", len(df))


# ==========================================
# STEP 2: CLEAN APPOINTMENT DATA
# ==========================================

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

print("After removing duplicates:", len(df))

# Clean booking channel
df["booking_channel"] = (
    df["booking_channel"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Convert date columns
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"], errors="coerce"
)

# Drop missing appointment date and status
df = df.dropna(
    subset=["appt_date", "status"]
).copy()

print("After date/status cleaning:", len(df))


# ==========================================
# STEP 3: VALIDATE AGE
# ==========================================

df["age"] = pd.to_numeric(
    df["age"], errors="coerce"
)

invalid_age = (
    (df["age"] < 1) | (df["age"] > 100)
)

df.loc[invalid_age, "age"] = np.nan

# Fill missing age with median
age_median = df["age"].median()

df["age"] = (
    df["age"]
    .fillna(age_median)
    .round()
    .astype(int)
)


# ==========================================
# STEP 4: FILL MISSING DISTANCE
# ==========================================

df["distance_km"] = pd.to_numeric(
    df["distance_km"], errors="coerce"
)

distance_median = df["distance_km"].median()

df["distance_km"] = df["distance_km"].fillna(
    distance_median
)


# ==========================================
# STEP 5: CAP DISTANCE OUTLIERS
# ==========================================

Q1_distance = df["distance_km"].quantile(0.25)
Q3_distance = df["distance_km"].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = (
    Q1_distance - 1.5 * IQR_distance
)

upper_distance = (
    Q3_distance + 1.5 * IQR_distance
)

df["distance_km"] = np.clip(
    df["distance_km"],
    lower_distance,
    upper_distance
)

df["distance_km"] = df["distance_km"].round(1)


# ==========================================
# STEP 6: CAP FEE OUTLIERS
# ==========================================

df["fee_inr"] = pd.to_numeric(
    df["fee_inr"], errors="coerce"
)

Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fee,
    upper_fee
)

df["fee_inr"] = (
    df["fee_inr"].round().astype("Int64")
)


# ==========================================
# STEP 7: CREATE LEAD_DAYS AND NO_SHOW
# ==========================================

df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

df["no_show"] = (
    df["status"] == "No-Show"
).astype(int)


# ==========================================
# STEP 8: SELECT EXACT 16 COLUMNS
# ==========================================

columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

df = df[columns].copy()


# ==========================================
# STEP 9: FORMAT DATE COLUMNS
# ==========================================

df["booked_on"] = (
    df["booked_on"].dt.strftime("%Y-%m-%d")
)

df["appt_date"] = (
    df["appt_date"].dt.strftime("%Y-%m-%d")
)


# ==========================================
# STEP 10: SORT BY APPOINTMENT ID
# ==========================================

df = df.sort_values(
    by="appointment_id"
).reset_index(drop=True)


# ==========================================
# STEP 11: VALIDATE FINAL DATA
# ==========================================

print("\n========== DATA VALIDATION ==========")

print("Final shape:", df.shape)

# Check appointment ID uniqueness
unique_ids = df["appointment_id"].is_unique

print("Appointment IDs unique:", unique_ids)

# Check missing values
missing_values = df.isna().sum()

print("\nMissing values in each column:")
print(missing_values)

if not unique_ids:
    raise ValueError("Duplicate appointment IDs found!")

if df.isna().any().any():
    raise ValueError(
        "Missing values found in appointment data. "
        "Check the columns printed above."
    )

if len(df.columns) != 16:
    raise ValueError("Expected exactly 16 columns!")


# ==========================================
# STEP 12: EXPORT CLEAN APPOINTMENT CSV
# ==========================================

clean_file = folder / "appointments_clean.csv"

df.to_csv(
    clean_file,
    index=False
)

print("\nClean CSV saved:", clean_file.name)


# ==========================================
# STEP 13: LOAD ENRICHED CLINIC DATA
# ==========================================

if not clinic_file.exists():
    raise FileNotFoundError(
        "clinics_enriched.csv not found. "
        "Run Task 1.5 first."
    )

clinics = pd.read_csv(clinic_file)

print("\nTotal clinics:", len(clinics))

if len(clinics) != 8:
    raise ValueError(
        "Expected exactly 8 clinic records!"
    )


# ==========================================
# STEP 14: CREATE SQL VALUE CONVERTER
# ==========================================

def sql_value(value):

    # Missing values become SQL NULL
    if pd.isna(value):
        return "NULL"

    # Convert NumPy values to Python values
    if isinstance(value, np.generic):
        value = value.item()

    # Numbers remain unquoted
    if isinstance(value, numbers.Number):
        return str(value)

    # Text values use single quotes
    text_value = str(value)

    return "'" + text_value.replace("'", "''") + "'"


# ==========================================
# STEP 15: GENERATE SQL SCRIPT
# ==========================================

sql_lines = []

# First insert all clinic records
print("\n========== GENERATING CLINIC SQL ==========")

for _, row in clinics.iterrows():

    values = [
        sql_value(value)
        for value in row.tolist()
    ]

    statement = (
        "INSERT INTO clinics VALUES ("
        + ", ".join(values)
        + ");"
    )

    sql_lines.append(statement)

print("Clinic SQL statements:", len(clinics))


# Then insert all appointment records
print("\n========== GENERATING APPOINTMENT SQL ==========")

for _, row in df.iterrows():

    values = [
        sql_value(value)
        for value in row.tolist()
    ]

    statement = (
        "INSERT INTO appointments VALUES ("
        + ", ".join(values)
        + ");"
    )

    sql_lines.append(statement)

print("Appointment SQL statements:", len(df))


# ==========================================
# STEP 16: SAVE SQL FILE
# ==========================================

sql_file = folder / "load_data.sql"

with open(
    sql_file,
    "w",
    encoding="utf-8"
) as file:

    file.write("\n".join(sql_lines))
    file.write("\n")


# ==========================================
# STEP 17: FINAL RESULTS
# ==========================================

print("\n========== TASK 1.6 FINAL RESULTS ==========")

print("Clean appointment rows:", len(df))
print("Appointment columns:", len(df.columns))

print("Unique appointment IDs:", unique_ids)

print("Missing appointment values:", df.isna().sum().sum())

print("Clinic SQL statements:", len(clinics))
print("Appointment SQL statements:", len(df))

print("Clean CSV:", clean_file.name)
print("SQL file:", sql_file.name)

print("\nTASK 1.6 COMPLETED SUCCESSFULLY!")

========== TASK 1.6 ==========
Original rows: 1020
After removing duplicates: 1000
After date/status cleaning: 990

========== DATA VALIDATION ==========
Final shape: (990, 16)
Appointment IDs unique: True

Missing values in each column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Clean CSV saved: appointments_clean.csv

Total clinics: 8

========== GENERATING CLINIC SQL ==========
Clinic SQL statements: 8

========== GENERATING APPOINTMENT SQL ==========
Appointment SQL statements: 990

========== TASK 1.6 FINAL RESULTS ==========
Clean appointment rows: 990
Appointment columns: 16
Unique appointment IDs: True
Missing appointment values: 0
Clinic SQL statements: 8
Appointment SQL stat